In [7]:
import pandas as pd
from pathlib import Path

listing_loc = pd.read_csv(Path("../data/processed/listings-loc.csv"), dtype = {"listing_id": "str"}, usecols = ["listing_id", "analysis_area"])
vung_loc = pd.read_csv(Path("../data/processed/vung-loc.csv"))
for i in ["lon_min", "lat_min", "lon_max", "lat_max"]:
    vung_loc[f"{i}"] = pd.to_numeric(vung_loc[f"{i}"].str.replace(",", ".", regex = False))
review_loc = pd.read_csv(Path("../data/processed/reviews-loc.csv"), usecols = ["listing_id", "review_id"], dtype = {"listing_id": "str", "review_id": "str"})

listing_path = Path("../data/raw/2026-03-21/listings.csv.gz")
listing = pd.read_csv(listing_path, compression= "gzip", dtype = {"id": "str", "host_id": "str"})
ls_ghep = listing.merge(listing_loc, left_on = "id", right_on = "listing_id", how = "right").merge(vung_loc, on = "analysis_area", how = "left")
ls_ghep["price"] = ls_ghep["price"].str.replace("$", "", regex = False).str.replace(",", "", regex = False)
ls_ghep["price_num"] = pd.to_numeric(ls_ghep["price"])


In [17]:
ls_ghep["lon_min"]

0     -0.495194
1     -0.075033
2     -0.192896
3     -0.137011
4     -0.137011
         ...   
995   -0.075033
996   -0.190431
997   -0.495194
998   -0.137011
999   -0.495194
Name: lon_min, Length: 1000, dtype: float64

In [ ]:
ls_ghep.to_csv(Path("../data/raw/ls_ghep.csv"), index=False)


In [16]:
def kiem_cau_truc(d):
    can = {"id", "price_num", "minimum_nights", "maximum_nights", "review_scores_rating"}
    thieu = can - set(d.columns)
    if thieu:
        raise ValueError(f"Thiếu cột: {sorted(thieu)}")
    for cot in ["price_num", "minimum_nights", "maximum_nights", "review_scores_rating"]:
        if not pd.api.types.is_numeric_dtype(d[cot]):
            raise TypeError(f"Cột {cot} phải có kiểu số")

kiem_cau_truc(ls_ghep)


In [7]:
import pandas as pd
from pathlib import Path
import yaml
ls = pd.read_csv(Path("../data/raw/ls_ghep.csv"))

In [1]:
def kiem_tra(d, nguong_gia):
    return pd.DataFrame({
        "flag_gia_thieu": d["price_num"].isna(),
        "gia_cao": d["price_num"] > nguong_gia,
        "dem_thieu": d["minimum_nights"].isna(),
        "dem_tren_365": d["minimum_nights"] > 365,
        "id_trung": d["id"].duplicated(keep=False),
    })


In [ ]:
df["price_clean"] = df.groupby(["room_type", "accommodates"])["price_num"].transform(
        lambda s: s.fillna(s.median())
    )